In [ ]:
import os
import warnings
import importlib

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
import sys
sys.path.append('/kaggle/input/datasets/keithmarange/hist-boost-methods/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [ ]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
model_run_name = 'cnn_1d_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

orientation_cols = [
    'Seated Straight',
    'Lie on Side - Non Dominant',
    'Seated Lean Non Dom - FACE DOWN',
    'Lie on Back'
]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report    = False
save_model   = False
random_search = False

In [ ]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=0.8,
    test_pct=0.2
)

# some_sequences = train_sample_df['sequence_id'].unique()[:50]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

In [ ]:
importlib.reload(utils)

num_pattern  = 'acc|rot|thm|tof'
suspect_cols = ['age', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
cat_cols     = ['orientation']
normal_cols  = ['adult_child', 'sex', 'handedness']
ordinal_cols = ['segment_id']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'feature_num_cols',
            StandardScaler(),
            make_column_selector(pattern=num_pattern)
        ),
        (
            'subject_num_cols',
            StandardScaler(),
            utils.existing_cols(suspect_cols)
        ),
        (
            'cat_encoder',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            utils.existing_cols(cat_cols)
        ),
        (
            'normal_cols',
            'passthrough',
            utils.existing_cols(normal_cols)
        ),
        (
            'ordinal_cols',
            'passthrough',
            utils.existing_cols(ordinal_cols)
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor.set_output(transform='pandas')

param_grid = {
    f'{pipe_name}__imu_sensor_list':        [None, acc_columns],
    f'{pipe_name}__imu_domain':             ['time', 'acceleration', 'velocity', 'displacement'],
    f'{pipe_name}__combine_imu_axes':       [True],
    f'{pipe_name}__sampling_rate':          [100],

    f'{pipe_name}__rotation_sensor_list':   [False, rot_columns],
    f'{pipe_name}__combine_rot_axes':       [True],
    f'{pipe_name}__rotation_domain':        ['orientation', 'motion', 'frequency'],

    f'{pipe_name}__thermopile_sensor_list': [None, thm_columns],
    f'{pipe_name}__thermopile_mode':        ['baseline', 'spatial'],
    f'{pipe_name}__tof_sensor_list':        [None, tof_columns],
    f'{pipe_name}__tof_mode':               ['baseline', 'blob', 'spatial'],

    f'{pipe_name}__window':                 [1.2],
    f'{pipe_name}__step_sec':               [0.2],

    f'{pipe_name}__dc_offset':              [0],
    f'{pipe_name}__band_edges':             [log_edges],
    f'{pipe_name}__category_data':          [False],
    f'{pipe_name}__segmentation':           ['window'],

    'pca__n_components': [None],

    'classifier__estimator__filters':       [32, 64],
    'classifier__estimator__kernel_size':   [3, 5],
    'classifier__estimator__dropout':       [0.3, 0.5],
    'classifier__estimator__learning_rate': [0.001],
    'classifier__estimator__epochs':        [30],
    'classifier__estimator__batch_size':    [32],
}

custom_extractor = utils.ImuExtractor(subject_df=train_demo_df)

In [ ]:
class Keras1DCNNClassifier(BaseEstimator, ClassifierMixin):
    """Sklearn-compatible wrapper around a Keras 1-D CNN classifier.

    Expects 2-D feature input (n_samples, n_features). Internally reshapes to
    (n_samples, n_features, 1) so that Conv1D treats each feature as a
    timestep with one channel. For proper temporal modelling, replace
    ImuExtractor with one that returns (n_samples, timesteps, channels) and
    adjust the Input shape accordingly.
    """

    def __init__(
        self,
        filters=32,
        kernel_size=3,
        dropout=0.3,
        learning_rate=0.001,
        epochs=30,
        batch_size=32,
        verbose=0
    ):
        self.filters       = filters
        self.kernel_size   = kernel_size
        self.dropout       = dropout
        self.learning_rate = learning_rate
        self.epochs        = epochs
        self.batch_size    = batch_size
        self.verbose       = verbose

    def _build_model(self, n_features, n_classes):
        model = keras.Sequential([
            layers.Input(shape=(n_features, 1)),
            layers.Conv1D(self.filters, self.kernel_size, activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.Conv1D(self.filters * 2, self.kernel_size, activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.GlobalAveragePooling1D(),
            layers.Dense(128, activation='relu'),
            layers.Dropout(self.dropout),
            layers.Dense(64, activation='relu'),
            layers.Dense(n_classes, activation='softmax'),
        ])
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=self.learning_rate),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

    def fit(self, X, y):
        if hasattr(X, 'values'):
            X = X.values
        self.le_ = LabelEncoder()
        y_enc = self.le_.fit_transform(y)
        self.classes_ = self.le_.classes_
        n_features = X.shape[1]
        n_classes  = len(self.classes_)
        X_3d = X.reshape(X.shape[0], n_features, 1)
        self.model_ = self._build_model(n_features, n_classes)
        self.model_.fit(
            X_3d, y_enc,
            epochs=self.epochs,
            batch_size=self.batch_size,
            verbose=self.verbose
        )
        return self

    def predict(self, X):
        if hasattr(X, 'values'):
            X = X.values
        X_3d = X.reshape(X.shape[0], X.shape[1], 1)
        probs = self.model_.predict(X_3d, verbose=0)
        return self.le_.inverse_transform(np.argmax(probs, axis=1))

    def predict_proba(self, X):
        if hasattr(X, 'values'):
            X = X.values
        X_3d = X.reshape(X.shape[0], X.shape[1], 1)
        return self.model_.predict(X_3d, verbose=0)

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


cnn_clf = Keras1DCNNClassifier(verbose=0)

In [ ]:
for model_target in model_target_list:

    cv_results_list = []

    for col in orientation_cols_dict:
        pipeline = Pipeline([
            (pipe_name, custom_extractor),
            ('preprocessor', preprocessor),
            ('pca', utils.IndexPreservingPCA()),
            ('classifier', utils.ManyToOneWrapper(
                estimator=cnn_clf,
                extractor=custom_extractor,
                mode=None,
                target=model_target
            ))
        ])

        search_obj = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=GroupKFold(n_splits=n_splits),
            verbose=1,
            n_jobs=1,          # n_jobs=1: Keras models are not picklable for parallel workers
            return_train_score=True
        )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        search_obj.fit(sliced_data_df, y, groups=sliced_data_df['sequence_id'])

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)

    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

In [ ]:
if do_report:

    best_model = search_obj.best_estimator_

    extractor    = best_model.named_steps['imu_extractor']
    preprocessor = best_model.named_steps['preprocessor']
    classifier   = best_model.named_steps['classifier']

    X_feat = extractor.transform(test_sample_df)
    X_proc = preprocessor.transform(X_feat)

    y_true = test_sample_df.drop_duplicates('sequence_id').set_index('sequence_id')['gesture']
    y_true = y_true.reindex(X_proc.index)

    y_pred = pd.Series(classifier.predict(X_proc), index=X_proc.index)

    print(classification_report(y_true, y_pred))

    report_df = pd.DataFrame(
        classification_report(y_true, y_pred, output_dict=True)
    ).T.sort_values('f1-score', ascending=False)

    report_df.to_csv(model_run_folder_name + 'cnn_1d_v1_per_gesture_scores.csv')